# Lab 01-10 · Multivariable Calculus II

Companion to chapter [01-10](../../chapters/01-10-multivariable-calculus-2.md). Handout: [README.md](README.md).

Run the cells in order. Cells marked **🔮 Predict** ask you to write a prediction *before* running.

In [1]:
import math, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import dblquad, tplquad, quad
sys.path.insert(0, str(Path.cwd().parents[3] / "tools"))
try:
    from plotstyle import use_style; use_style()
except ImportError:
    pass
rng = np.random.default_rng(0)

## Part 1 — Multiple integrals: quadrature against Monte Carlo

**🔮 Predict:** how much does the Monte Carlo error shrink when the number of samples grows 100×?

In [2]:
p = lambda y, x: 0.2 + 0.02 * x * y             # die power density, W/mm^2 (Example 01-10.1)
P_quad, err = dblquad(p, 0, 10, 0, 10)
print(f"dblquad: total power = {P_quad:.6f} W (estimated error {err:.1e})")
for N in [100, 10_000, 1_000_000]:
    errs = []
    for _ in range(50):
        xs, ys = rng.uniform(0, 10, N), rng.uniform(0, 10, N)
        errs.append(100 * np.mean(0.2 + 0.02 * xs * ys) - 70)
    print(f"Monte Carlo, N = {N:>9,}: RMS error over 50 runs = {np.sqrt(np.mean(np.square(errs))):.4f} W")

rho0, R, eps0 = 1e-6, 0.1, 8.8541878128e-12
Q, _ = tplquad(lambda r, ph, th: rho0 * (1 - r / R) * r**2 * math.sin(ph), 0, 2 * math.pi, 0, math.pi, 0, R)
print(f"charged sphere (Example 01-10.5): Q = {Q:.4e} C, E(R) = {Q/(4*math.pi*eps0*R*R):.1f} V/m")

dblquad: total power = 70.000000 W (estimated error 7.8e-13)
Monte Carlo, N =       100: RMS error over 50 runs = 4.9931 W
Monte Carlo, N =    10,000: RMS error over 50 runs = 0.4023 W


Monte Carlo, N = 1,000,000: RMS error over 50 runs = 0.0383 W
charged sphere (Example 01-10.5): Q = 1.0472e-09 C, E(R) = 941.2 V/m


## Part 2 — Choosing coordinates: a Gaussian over a disk

In [3]:
exact = math.pi * (1 - math.exp(-4))            # ∬_{r<=2} e^{-r^2} dA
for n in [10, 40, 160]:
    # Cartesian midpoint grid on the square [-2, 2]^2, masked to the disk
    g = np.linspace(-2, 2, n + 1); mid = (g[:-1] + g[1:]) / 2; h = 4 / n
    X, Y = np.meshgrid(mid, mid); inside = X**2 + Y**2 <= 4
    cart = np.sum(np.exp(-(X**2 + Y**2))[inside]) * h * h
    # polar midpoint rule in r (theta integral is exact by symmetry)
    rr = (np.arange(n) + 0.5) * (2 / n)
    polar = 2 * math.pi * np.sum(np.exp(-rr**2) * rr) * (2 / n)
    print(f"n = {n:>4}: Cartesian error {abs(cart-exact):.2e} with {n*n:>6} evaluations;"
          f"  polar error {abs(polar-exact):.2e} with {n:>4} evaluations")

n =   10: Cartesian error 5.02e-03 with    100 evaluations;  polar error 1.19e-02 with   10 evaluations
n =   40: Cartesian error 1.36e-03 with   1600 evaluations;  polar error 7.39e-04 with   40 evaluations
n =  160: Cartesian error 2.09e-05 with  25600 evaluations;  polar error 4.62e-05 with  160 evaluations


For similar accuracy, the Cartesian grid needs $n^2$ evaluations and the polar one only $n$: symmetry turned a two-dimensional integral into a one-dimensional one. The Cartesian error also jumps around irregularly, because squares cut by the circular boundary are either counted whole or dropped; the polar grid fits the boundary exactly.

## Part 3 — Line integrals and Green's theorem

In [4]:
def line_integral(F, path, n=20001):
    t = np.linspace(0, 1, n)
    x, y = path(t)
    dx, dy = np.gradient(x, t), np.gradient(y, t)
    Fx, Fy = F(x, y)
    return np.trapezoid(Fx * dx + Fy * dy, t)

F_cons = lambda x, y: (2 * x * y, x**2)        # gradient of x^2 y
straight = lambda t: (t, 2 * t)
curvy = lambda t: (t**3, 2 * np.sin(np.pi * t / 2))
print(f"conservative field, (0,0)->(1,2): straight {line_integral(F_cons, straight):.6f}, curvy {line_integral(F_cons, curvy):.6f}")

G = lambda x, y: (-y, x)
circle = lambda t: (2 * np.cos(2 * np.pi * t), 2 * np.sin(2 * np.pi * t))
print(f"circulation of (-y, x) around r = 2: {line_integral(G, circle):.6f}  (Green: 2 x area = {8*math.pi:.6f})")
W = lambda x, y: (-y / (x**2 + y**2), x / (x**2 + y**2))
unit = lambda t: (np.cos(2 * np.pi * t), np.sin(2 * np.pi * t))
shifted = lambda t: (3 + np.cos(2 * np.pi * t), np.sin(2 * np.pi * t))
print(f"wire-like field: around the origin {line_integral(W, unit):.6f}, around a loop not enclosing it {line_integral(W, shifted):.2e}")

conservative field, (0,0)->(1,2): straight 2.000000, curvy 2.000000
circulation of (-y, x) around r = 2: 25.132741  (Green: 2 x area = 25.132741)
wire-like field: around the origin 6.283185, around a loop not enclosing it 7.77e-16


## Part 4 — Counting dies on a wafer

**🔮 Predict:** for 25 mm dies, will the count of whole dies depend on where the grid is placed? By how much?

In [5]:
def whole_dies(d, s, ox=0.0, oy=0.0):
    count = 0
    for i in np.arange(-d / 2 - s, d / 2 + s, s) + ox:
        for j in np.arange(-d / 2 - s, d / 2 + s, s) + oy:
            corners = [(i, j), (i + s, j), (i, j + s), (i + s, j + s)]
            if all(math.hypot(cx, cy) <= d / 2 for cx, cy in corners):
                count += 1
    return count

d = 300.0
for s in [10.0, 25.0]:
    A = s * s
    formula = math.pi * (d / 2)**2 / A - math.pi * d / math.sqrt(2 * A)
    counts = [whole_dies(d, s, ox, oy) for ox in np.linspace(0, s, 6, endpoint=False) for oy in np.linspace(0, s, 6, endpoint=False)]
    print(f"die {s:.0f} mm: formula {formula:.1f}; grid counts from {min(counts)} to {max(counts)} (centered grid: {whole_dies(d, s)})")

die 10 mm: formula 640.2; grid counts from 640 to 657 (centered grid: 648)
die 25 mm: formula 86.4; grid counts from 86 to 93 (centered grid: 88)


## Your turn

1. In Part 1, estimate the charged-sphere integral by Monte Carlo. (Hint: sample points uniformly in a cube and keep those inside the sphere.) How many samples give 0.1% accuracy?
2. In Part 4, find the grid offset that maximizes whole 25 mm dies. Fabs and design teams optimize exactly this, the "wafer map", for large dies.